In [5]:
# prepare data
import pickle 
import numpy as np
with open("./release_assets/metadata/new_hsp60/centers3.pkl", "rb") as f:
    hsp60_centers = pickle.load(f)

In [10]:
hsp60_centers
for i in hsp60_centers:
    array = np.array(hsp60_centers[i]) 
    array = array[:, [2, 1, 0]]
    np.savetxt(f"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/raw_data/{i}.txt", array, delimiter="\t")

In [11]:
array = np.loadtxt("/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/raw_data/24092001.txt", delimiter="\t")
a2 = np.loadtxt("/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/raw_data/coords/24092001.coords", delimiter="\t")
_, match_distances, closest_preds = match_and_find_closest(a2, array)
print("Matched distances:", match_distances)

Matched distances: [1.39962167e+00 1.03307728e+00 2.98679555e+01 1.73353758e+01
 2.10013021e+01 2.23418873e+00 5.33225042e+01 1.42198209e+00
 2.00470908e+00 9.91395149e-01 9.83853926e-01 1.99066939e+00
 5.38922674e+00 1.41871702e+00 9.95455314e-01 2.45751203e+00
 1.97439070e+00 1.43124997e+00 1.44201719e+00 4.12549528e-02
 1.01403732e+00 1.73425353e+00 1.72811943e+00 1.00571937e+00
 2.47038078e+00 3.15930061e+00 1.39940805e+00 1.41197885e+00
 1.03405590e+00 1.13716133e+02 3.24760450e-02 3.16428931e+00
 1.00366163e+00 1.20047424e+02 1.08069459e+02 3.19852850e+00
 1.40664492e+00 1.41049698e+00 1.42011837e+00 2.84639523e-02
 1.41599001e+00 1.72186005e+00 2.22703887e+00 2.00352351e+00
 1.41487240e+00 9.93379675e-01 1.42688304e+00 2.47490069e+00
 1.42603651e+00 1.40673591e+00 1.72639834e+00 2.24452591e+01
 3.17510159e+00 7.74386646e+01 2.00865420e+00 2.27031236e+00
 1.77684864e+00 5.72381869e+01 1.72325925e+00 2.89047157e+01
 2.43340179e+00 5.05026789e+01 1.02443224e+00 5.96095543e+01
 1.75

In [ ]:
## DeepETPicker hsp60 run

This section uses the six hsp60 tomograms in `raw_data/`.
- Train: `24092001`, `24092002`, `24092018`, `24092024`, `24092028`
- Hold out for testing: `24092031`

The workflow follows the repo docs: preprocessing, train config generation, training, then testing.

In [17]:
from pathlib import Path
import json
import subprocess
import sys

repo = Path("./release_assets/external/DeepETPicker")
base = Path("/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60")
raw = base / "raw_data"
coords_dir = raw / "coords"
cfg_dir = base / "configs"
cfg_dir.mkdir(parents=True, exist_ok=True)
runner_python = Path("python")

train_ids = "0,1,2,3,4"
val_id = "5"

# Keep the notebook split explicit so the training and test steps are reproducible.
tomos = sorted(p.stem for p in raw.glob("*.mrc"))
coords = sorted(p.stem for p in coords_dir.glob("*.coords"))
print("Tomograms:", tomos)
print("Coords:", coords)
assert tomos == coords, "Tomogram and coord file stems do not match"

pre_config = {
    "dset_name": "hsp60_preprocess",
    "base_path": str(base),
    "coord_path": str(coords_dir),
    "coord_format": ".coords",
    "tomo_path": str(raw),
    "tomo_format": ".mrc",
    "num_cls": 1,
    "label_type": "gaussian",
    "label_diameter": 11,
    "ocp_type": "sphere",
    "ocp_diameter": "23",
    "norm_type": "standardization",
}

pre_cfg_path = base / "hsp60_preprocess.py"
pre_cfg_path.write_text("pre_config=" + json.dumps(pre_config, indent=12))
print(f"Saved preprocessing config to {pre_cfg_path}")
print(f"Train ids: {train_ids}")
print(f"Val/Test id: {val_id}")

def run_cmd(cmd, log_name):
    log_path = base / log_name
    with log_path.open("w") as log_file:
        subprocess.run(
            cmd,
            cwd=repo / "bin",
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=True,
        )
    tail = log_path.read_text().splitlines()[-20:]
    print(f"Finished {log_name}")
    if tail:
        print("\n".join(tail))
    return log_path

Tomograms: ['24092001', '24092002', '24092018', '24092024', '24092028', '24092031']
Coords: ['24092001', '24092002', '24092018', '24092024', '24092028', '24092031']
Saved preprocessing config to /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/hsp60_preprocess.py
Train ids: 0,1,2,3,4
Val/Test id: 5


In [11]:
run_cmd([runner_python, str(repo / "bin" / "preprocess.py"), "--pre_configs", str(pre_cfg_path)], "preprocess.log")

Finished preprocess.log
0 to 2: 477
0 to 3: 633
0 to 4: 810
0 to 5: 866
0 to 6: 1008
Coord generation finished!
****************************************************************************************************
['24092001.mrc', '24092002.mrc', '24092018.mrc', '24092024.mrc', '24092028.mrc', '24092031.mrc']
4/6 finished.
3/6 finished.
2/6 finished.
5/6 finished.
6/6 finished.
1/6 finished.
Standardization finished!
****************************************************************************************************
Label generation finished!
****************************************************************************************************
Occupancy generation finished!
****************************************************************************************************


PosixPath('/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/preprocess.log')

In [18]:
run_cmd([
    runner_python,
    str(repo / "bin" / "generate_train_config.py"),
    "--pre_configs",
    str(pre_cfg_path),
    "--dset_name",
    "hsp60_train",
    "--cfg_save_path",
    str(cfg_dir),
    "--train_set_ids",
    train_ids,
    "--val_set_ids",
    val_id,
    "--batch_size",
    "8",
    "--block_size",
    "72",
    "--pad_size",
    "12",
    "--learning_rate",
    "0.001",
    "--max_epoch",
    "60",
    "--threshold",
    "0.5",
    "--gpu_id",
    "0",
], "generate_train_config.log")
train_cfg_path = cfg_dir / "hsp60_train.py"
print(f"Training config: {train_cfg_path}")
print(train_cfg_path.read_text().splitlines()[0])

Finished generate_train_config.log
0
Training config: /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/configs/hsp60_train.py
train_configs={"dset_name": "hsp60_train",


In [24]:
run_cmd([runner_python, str(repo / "bin" / "train_bash.py"), "--train_configs", str(train_cfg_path)], "train.log")

Finished train.log

Validation DataLoader 0:  78%|███████▊  | 14/18 [00:01<00:00, 11.96it/s]

Validation DataLoader 0:  83%|████████▎ | 15/18 [00:01<00:00, 12.07it/s]

Validation DataLoader 0:  89%|████████▉ | 16/18 [00:01<00:00, 12.18it/s]

Validation DataLoader 0:  94%|█████████▍| 17/18 [00:01<00:00, 12.26it/s]

Validation DataLoader 0: 100%|██████████| 18/18 [00:01<00:00, 12.47it/s]

                                                                        
Epoch 59: 100%|██████████| 109/109 [00:32<00:00,  3.31it/s, v_num=3]`Trainer.fit` stopped: `max_epochs=60` reached.

Epoch 59: 100%|██████████| 109/109 [00:32<00:00,  3.31it/s, v_num=3]
****************************************************************************************************
Training Finished
Training pid:2692568
****************************************************************************************************


PosixPath('/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/train.log')

In [29]:
ckpt_candidates = sorted((base / "runs" / "hsp60_train").glob("**/*.ckpt"), key=lambda p: p.stat().st_mtime)
print("Checkpoint candidates:")
for candidate in ckpt_candidates:
    print(candidate)
assert ckpt_candidates, "No checkpoint was produced by training"
best_ckpt = ckpt_candidates[-1]
print(f"Using checkpoint: {best_ckpt}")

run_cmd([
    runner_python,
    str(repo / "bin" / "test_bash.py"),
    "--train_configs",
    str(train_cfg_path),
    "--checkpoints",
    str(best_ckpt),
], "test.log")

Checkpoint candidates:
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train/hsp60_train_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_2/checkpoints/epoch=15-step=1744.ckpt
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train/hsp60_train_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_3/checkpoints/epoch=7-step=872.ckpt
Using checkpoint: /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train/hsp60_train_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_3/checkpoints/epoch=7-step=872.ckpt
Finished test.log
Testing DataLoader 0: 100%|██████████| 666/666 [00:46<00:00, 14.45it/s]
****************************************************************************************************
Testing Finished!
****************************************************************************************************


PosixPath('/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/test.log')

In [30]:
import csv
import time
from datetime import datetime


def run_cmd_capture(cmd, log_name):
    log_path = base / log_name
    start = time.time()
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo / "bin",
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )
    duration_sec = round(time.time() - start, 2)
    lines = log_path.read_text().splitlines()
    tail = lines[-20:]
    print(f"[{log_name}] exit={proc.returncode}, duration={duration_sec}s")
    if tail:
        print("\n".join(tail))
    return proc.returncode, log_path, duration_sec


all_ids = list(range(len(tomos)))
existing_holdout = int(val_id)
holdouts_to_run = [idx for idx in all_ids if idx != existing_holdout]

results = []

for holdout in holdouts_to_run:
    train_ids_loop = ",".join(str(i) for i in all_ids if i != holdout)
    val_id_loop = str(holdout)
    dset_name_loop = f"hsp60_train_holdout_{holdout}"
    train_cfg_loop = cfg_dir / f"{dset_name_loop}.py"

    print("=" * 100)
    print(f"Running split: holdout={holdout}, test_tomo={tomos[holdout]}, train_ids={train_ids_loop}")

    gen_cmd = [
        runner_python,
        str(repo / "bin" / "generate_train_config.py"),
        "--pre_configs",
        str(pre_cfg_path),
        "--dset_name",
        dset_name_loop,
        "--cfg_save_path",
        str(cfg_dir),
        "--train_set_ids",
        train_ids_loop,
        "--val_set_ids",
        val_id_loop,
        "--batch_size",
        "8",
        "--block_size",
        "72",
        "--pad_size",
        "12",
        "--learning_rate",
        "0.001",
        "--max_epoch",
        "60",
        "--threshold",
        "0.5",
        "--gpu_id",
        "0",
    ]
    gen_code, gen_log, gen_sec = run_cmd_capture(gen_cmd, f"generate_train_config_holdout_{holdout}.log")

    split_result = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "holdout_id": holdout,
        "holdout_tomogram": tomos[holdout],
        "train_ids": train_ids_loop,
        "val_id": val_id_loop,
        "dset_name": dset_name_loop,
        "train_config": str(train_cfg_loop),
        "generate_log": str(gen_log),
        "generate_exit": gen_code,
        "generate_seconds": gen_sec,
        "train_log": "",
        "train_exit": None,
        "train_seconds": None,
        "checkpoint": "",
        "test_log": "",
        "test_exit": None,
        "test_seconds": None,
        "status": "pending",
    }

    if gen_code != 0:
        split_result["status"] = "generate_failed"
        results.append(split_result)
        continue

    train_cmd = [
        runner_python,
        str(repo / "bin" / "train_bash.py"),
        "--train_configs",
        str(train_cfg_loop),
    ]
    train_code, train_log, train_sec = run_cmd_capture(train_cmd, f"train_holdout_{holdout}.log")
    split_result["train_log"] = str(train_log)
    split_result["train_exit"] = train_code
    split_result["train_seconds"] = train_sec

    if train_code != 0:
        split_result["status"] = "train_failed"
        results.append(split_result)
        continue

    ckpt_candidates = sorted((base / "runs" / dset_name_loop).glob("**/*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpt_candidates:
        split_result["status"] = "no_checkpoint"
        results.append(split_result)
        continue

    best_ckpt_loop = ckpt_candidates[-1]
    split_result["checkpoint"] = str(best_ckpt_loop)
    print(f"Using checkpoint: {best_ckpt_loop}")

    test_cmd = [
        runner_python,
        str(repo / "bin" / "test_bash.py"),
        "--train_configs",
        str(train_cfg_loop),
        "--checkpoints",
        str(best_ckpt_loop),
    ]
    test_code, test_log, test_sec = run_cmd_capture(test_cmd, f"test_holdout_{holdout}.log")
    split_result["test_log"] = str(test_log)
    split_result["test_exit"] = test_code
    split_result["test_seconds"] = test_sec
    split_result["status"] = "ok" if test_code == 0 else "test_failed"

    results.append(split_result)

results_json = base / "leave_one_out_results_5_splits.json"
results_json.write_text(json.dumps(results, indent=2))

results_csv = base / "leave_one_out_results_5_splits.csv"
with results_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()) if results else [
        "timestamp", "holdout_id", "holdout_tomogram", "train_ids", "val_id", "dset_name",
        "train_config", "generate_log", "generate_exit", "generate_seconds", "train_log", "train_exit",
        "train_seconds", "checkpoint", "test_log", "test_exit", "test_seconds", "status"
    ])
    writer.writeheader()
    writer.writerows(results)

print("\nSaved summary files:")
print(results_json)
print(results_csv)
print("\nResults:")
for r in results:
    print(r)

Running split: holdout=0, test_tomo=24092001, train_ids=1,2,3,4,5
[generate_train_config_holdout_0.log] exit=0, duration=0.14s
0
[train_holdout_0.log] exit=0, duration=1777.78s

Validation DataLoader 0:  90%|████████▉ | 35/39 [00:02<00:00, 13.13it/s]

Validation DataLoader 0:  92%|█████████▏| 36/39 [00:02<00:00, 13.16it/s]

Validation DataLoader 0:  95%|█████████▍| 37/39 [00:02<00:00, 13.18it/s]

Validation DataLoader 0:  97%|█████████▋| 38/39 [00:02<00:00, 13.20it/s]

Validation DataLoader 0: 100%|██████████| 39/39 [00:02<00:00, 13.26it/s]

                                                                        
Epoch 59: 100%|██████████| 88/88 [00:27<00:00,  3.18it/s, v_num=0]`Trainer.fit` stopped: `max_epochs=60` reached.

Epoch 59: 100%|██████████| 88/88 [00:27<00:00,  3.18it/s, v_num=0]
****************************************************************************************************
Training Finished
Training pid:2746317
*********************************************************

In [34]:
import json
import csv
import numpy as np
from pathlib import Path

# Matching tolerance in voxels for TP assignment.
match_threshold = 10.0


def _load_coords(path):
    arr = np.loadtxt(path, delimiter="\t")
    arr = np.atleast_2d(arr)
    if arr.shape[1] >= 3:
        return arr[:, :3].astype(float)
    raise ValueError(f"Unexpected coord format in {path}: shape={arr.shape}")


def greedy_match_f1(gt_xyz, pred_xyz, threshold):
    if len(gt_xyz) == 0 and len(pred_xyz) == 0:
        return {"tp": 0, "fp": 0, "fn": 0, "precision": 1.0, "recall": 1.0, "f1": 1.0}
    if len(gt_xyz) == 0:
        fp = len(pred_xyz)
        return {"tp": 0, "fp": fp, "fn": 0, "precision": 0.0, "recall": 1.0, "f1": 0.0}
    if len(pred_xyz) == 0:
        fn = len(gt_xyz)
        return {"tp": 0, "fp": 0, "fn": fn, "precision": 1.0, "recall": 0.0, "f1": 0.0}

    pairs = []
    for pi, p in enumerate(pred_xyz):
        d = np.linalg.norm(gt_xyz - p, axis=1)
        gi = int(np.argmin(d))
        md = float(d[gi])
        if md <= threshold:
            pairs.append((md, pi, gi))

    pairs.sort(key=lambda x: x[0])
    used_p = set()
    used_g = set()
    tp = 0
    for _, pi, gi in pairs:
        if pi in used_p or gi in used_g:
            continue
        used_p.add(pi)
        used_g.add(gi)
        tp += 1

    fp = len(pred_xyz) - tp
    fn = len(gt_xyz) - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def _coords_dirs_for_ckpt(ckpt_path):
    version_dir = Path(ckpt_path).parents[1]
    return [
        version_dir / "PredictedLabels" / "Coords_All",
        version_dir / "TestRes" / "Coords_All",
    ]


def run_test_for_split(split_name, train_cfg, ckpt_path):
    log_name = f"f1_test_{split_name}.log"
    log_path = base / log_name

    dirs = _coords_dirs_for_ckpt(ckpt_path)
    before = {}
    for d in dirs:
        before[d] = set(d.glob("*.coords")) if d.exists() else set()

    cmd = [
        runner_python,
        str(repo / "bin" / "test_bash.py"),
        "--train_configs",
        str(train_cfg),
        "--checkpoints",
        str(ckpt_path),
    ]
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo / "bin",
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )
    if proc.returncode != 0:
        tail = "\n".join(log_path.read_text().splitlines()[-40:])
        raise RuntimeError(f"Test failed for {split_name} (exit={proc.returncode})\n{tail}")

    # Prefer newly created coords file; if overwritten, use newest by mtime.
    candidates_new = []
    candidates_all = []
    for d in dirs:
        if not d.exists():
            continue
        now = set(d.glob("*.coords"))
        candidates_new.extend(list(now - before.get(d, set())))
        candidates_all.extend(list(now))

    if candidates_new:
        pred_path = max(candidates_new, key=lambda p: p.stat().st_mtime)
    elif candidates_all:
        pred_path = max(candidates_all, key=lambda p: p.stat().st_mtime)
    else:
        raise FileNotFoundError(f"No prediction coords generated for {split_name}")

    return pred_path, log_path


# Build split metadata for all 6 tests.
all_splits = []

# Existing holdout-5 run from hsp60_train.
base_cfg = cfg_dir / "hsp60_train.py"
base_ckpts = sorted((base / "runs" / "hsp60_train").glob("**/*.ckpt"), key=lambda p: p.stat().st_mtime)
assert base_ckpts, "No checkpoint found under runs/hsp60_train"
all_splits.append({
    "split_name": "holdout_5",
    "holdout_id": 5,
    "train_config": base_cfg,
    "checkpoint": base_ckpts[-1],
})

# Holdout 0..4 runs from previous batch results.
loo_results = json.loads((base / "leave_one_out_results_5_splits.json").read_text())
for r in loo_results:
    hid = int(r["holdout_id"])
    all_splits.append({
        "split_name": f"holdout_{hid}",
        "holdout_id": hid,
        "train_config": Path(r["train_config"]),
        "checkpoint": Path(r["checkpoint"]),
    })

all_splits = sorted(all_splits, key=lambda x: x["holdout_id"])

f1_rows = []
for s in all_splits:
    pred_path, test_log_path = run_test_for_split(
        split_name=s["split_name"],
        train_cfg=s["train_config"],
        ckpt_path=s["checkpoint"],
    )

    holdout_tomo = pred_path.stem
    gt_path = raw / "coords" / f"{holdout_tomo}.coords"
    if not gt_path.exists():
        raise FileNotFoundError(f"Ground truth coords not found for {holdout_tomo}: {gt_path}")

    gt_xyz = _load_coords(gt_path)
    pred_xyz = _load_coords(pred_path)
    m = greedy_match_f1(gt_xyz, pred_xyz, match_threshold)

    row = {
        "split_name": s["split_name"],
        "holdout_id": s["holdout_id"],
        "holdout_tomogram": holdout_tomo,
        "train_config": str(s["train_config"]),
        "checkpoint": str(s["checkpoint"]),
        "prediction_coords": str(pred_path),
        "test_log": str(test_log_path),
        "gt_count": int(len(gt_xyz)),
        "pred_count": int(len(pred_xyz)),
        "tp": m["tp"],
        "fp": m["fp"],
        "fn": m["fn"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "match_threshold_voxels": match_threshold,
    }
    f1_rows.append(row)

# Micro-average across all 6 tests.
tp_sum = sum(r["tp"] for r in f1_rows)
fp_sum = sum(r["fp"] for r in f1_rows)
fn_sum = sum(r["fn"] for r in f1_rows)
precision_micro = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) > 0 else 0.0
recall_micro = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) > 0 else 0.0
f1_micro = (2 * precision_micro * recall_micro / (precision_micro + recall_micro)) if (precision_micro + recall_micro) > 0 else 0.0

f1_json = base / "f1_results_all_6_tests.json"
f1_csv = base / "f1_results_all_6_tests.csv"

f1_json.write_text(json.dumps({
    "match_threshold_voxels": match_threshold,
    "micro_avg": {
        "tp": int(tp_sum),
        "fp": int(fp_sum),
        "fn": int(fn_sum),
        "precision": float(precision_micro),
        "recall": float(recall_micro),
        "f1": float(f1_micro),
    },
    "per_split": f1_rows,
}, indent=2))

with f1_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(f1_rows[0].keys()))
    writer.writeheader()
    writer.writerows(f1_rows)

print("Per-split F1:")
for r in f1_rows:
    print(
        f"{r['split_name']} ({r['holdout_tomogram']}): "
        f"P={r['precision']:.4f}, R={r['recall']:.4f}, F1={r['f1']:.4f}, "
        f"TP/FP/FN={r['tp']}/{r['fp']}/{r['fn']}"
    )

print("\nMicro-average over all 6 tests:")
print(
    f"P={precision_micro:.4f}, R={recall_micro:.4f}, F1={f1_micro:.4f}, "
    f"TP/FP/FN={tp_sum}/{fp_sum}/{fn_sum}"
)

print("\nSaved:")
print(f1_json)
print(f1_csv)

Per-split F1:
holdout_0 (24092018): P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/478/156
holdout_1 (24092028): P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/227/56
holdout_2 (24092024): P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/583/177
holdout_3 (24092031): P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/517/142
holdout_4 (24092002): P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/398/166
holdout_5 (24092001): P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/587/311

Micro-average over all 6 tests:
P=0.0000, R=0.0000, F1=0.0000, TP/FP/FN=0/2790/1008

Saved:
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/f1_results_all_6_tests.json
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/f1_results_all_6_tests.csv


In [37]:
import os
for s in all_splits:
    pred_path, test_log_path = run_test_for_split(
        split_name=s["split_name"],
        train_cfg=s["train_config"],
        ckpt_path=s["checkpoint"],
    )
    print(pred_path, test_log_path)
    os.system(f"cp {pred_path} /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60_new")

/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train_holdout_0/hsp60_train_holdout_0_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_0/PredictedLabels/Coords_All/24092018.coords /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/f1_test_holdout_0.log
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train_holdout_1/hsp60_train_holdout_1_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_0/PredictedLabels/Coords_All/24092028.coords /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/f1_test_holdout_1.log
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train_holdout_2/hsp60_train_holdout_2_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_0/PredictedLabels/Coords_All/24092024.coords /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/f1_test_holdout_2.log
/virtual_data/test_data/dxd20240920/D

In [ ]:
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/

In [36]:
import pandas as pd
t = "/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train_holdout_0/hsp60_train_holdout_0_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_0/PredictedLabels/Coords_All/24092018.coords"
res_18 = pd.read_csv(t, delimiter="\t", header=None)
res_18.columns = ["x", "y", "z"]

ModuleNotFoundError: No module named 'pandas'